# **Introduction to FlyWire — Part 2: Synapses and connections**

In Part 1 we looked at the giant fiber (GF, *DNp01*), the descending neuron that triggers the fly's escape take-off. We found its root ID, looked at its shape, and read its annotations. But a connectome is more than a collection of shapes: it's the wiring diagram In this notebook we'll answer one concrete question: **Who talks to the giant fiber, and how?**

Along the way you'll learn how FlyWire represents synapses, how to turn individual synapses into neuron-to-neuron connections, and how to summarize connectivity by cell type.

## 0. Setup and recap
As in Part 1: [click here](colab.research.google.com/github/ryanrahy/fcnc_workshop/blob/master/day_1/FW_intro_part2_connections.ipynb) to run this notebook on Colab.

In [ ]:
# run this cell if on a fresh machine or on Colab

# get code + data
!git clone https://github.com/ryanrahy/fcnc_workshop.git
import os
os.chdir('./fcnc_workshop/day1/')

# install missing packages
!pip install navis flybrains ipywidgets plotly pyarrow
!pip install fafbseg   # only needed for online mode (see below)

Choose **offline mode** (pre-downloaded data, no token) or **online mode** (live queries, needs `fafbseg` and a FlyWire token, see Part 1, Section 0).

In [ ]:
OFFLINE = True   # set to False to query FlyWire live

if OFFLINE:
    import fw_offline as flywire
else:
    from fafbseg import flywire

import navis
navis.set_loggers("ERROR")  # hide navis' chatty warnings (e.g. about soma detection)
import flybrains

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

flywire.set_default_dataset("public")
NC = flywire.NeuronCriteria

# Our protagonists (root IDs at materialization 783)
GF_LEFT = 720575940622838154
GF_RIGHT = 720575940632499757

# Colors we'll use consistently for the two main input types
COLORS = {"LC4": "#2a78d6", "LPLC2": "#eb6834"}
GRAY = "#b4b2a9"

Connectivity tables only contain root IDs, which are not very informative on their own. So we load the annotation table for all neurons once and build a lookup from **root ID → cell type**. (If a neuron has no `cell_type`, we fall back to its `hemibrain_type`.)

In [ ]:
ann = flywire.search_annotations(None)
# Use each neuron's cell type, or its hemibrain type if it has no cell type
ann["label"] = ann.cell_type.fillna(ann.hemibrain_type)

# Lookup table: root ID → label (one row per neuron)
type_of = ann.drop_duplicates("root_id").set_index("root_id")["label"]
type_of

In [ ]:
type_of[[GF_LEFT, GF_RIGHT]]

---
## 1. Synapses: the raw material

FlyWire's synapses were detected automatically by a neural network trained on the EM images (Buhmann et al., 2021). Each detected synapse has:

* a **presynaptic** neuron (`pre`, the sender) and a **postsynaptic** neuron (`post`, the receiver),
* a position on each side (`pre_x/y/z`, `post_x/y/z`, in **nm**),
* a `cleft_score`, which measures how confident the detector is. By default fafbseg uses the *filtered* table (score > 50, duplicates removed), the same as the FlyWire papers.

### 1.1 All synapses of one neuron
Getting all the synapses that are associated with a single neuron:

In [ ]:
synapses = flywire.get_synapses(GF_LEFT)
print(f"{len(synapses):,} synapses")
synapses.head()

Every row is **one synapse**. Whether it is an input or an output depends on which column our neuron appears in:

In [ ]:
inputs = synapses[synapses.post == GF_LEFT]    # GF receives → input
outputs = synapses[synapses.pre == GF_LEFT]    # GF sends    → output

print(f"inputs (GF is postsynaptic): {len(inputs):>6,}")
print(f"outputs (GF is presynaptic): {len(outputs):>6,}")

### 1.2 Synapses on the neuron
Instead of abstractly querying synapses, we can also associate them directly with a neuron skeleton for plotting or analysis. With `attach=True`, each synapse is mapped to the closest skeleton node and stored in the neuron's `.connectors` table:

In [ ]:
# Fetch a skeleton (and fix the soma, see Part 1)
sk = flywire.get_skeletons(GF_LEFT)
sk.soma = sk.root[0]

# Get the synapses associated with that neuron and attach them to the skeleton
flywire.get_synapses(sk, attach=True)

# Now the neuron has a connector table
sk.connectors.head()

In the connector table, `type` is from the neuron's point of view: **`pre`** means our neuron is presynaptic (an **output**), and **`post`** means it is postsynaptic (an **input**). `partner_id` is the neuron on the other side.

Let's plot where the inputs and outputs are:

In [ ]:
# doing it directly with navis
fig, ax = navis.plot2d(sk, connectors = True, color = 'gray', lw = 0.5, view = ('x', '-y'))

In [ ]:
cn_tab = sk.connectors
fig, ax = navis.plot2d(sk, color=GRAY, lw=0.5, view=("x", "-y"))
handles = []
for kind, label, color in [("post", "inputs", "#2a78d6"), ("pre", "outputs", "#eb6834")]:
    s = cn_tab[cn_tab.type == kind]
    h = ax.scatter(s.x, s.y, s=1, color=color, label=f"{label} ({len(s):,})", zorder=10, rasterized=True)
    handles.append(h)
ax.legend(handles=handles, markerscale=8, frameon=False, loc="lower left")
ax.set_title("Left giant fiber: synapse locations (frontal view)");

In [ ]:
# the same plot in 3D
import plotly.graph_objects as go

cn_tab = sk.connectors
fig = navis.plot3d(sk, color=GRAY, backend="plotly", inline=False)  # returns a plotly Figure

for kind, label, color in [("post", "inputs", "#2a78d6"), ("pre", "outputs", "#eb6834")]:
    s = cn_tab[cn_tab.type == kind]
    fig.add_trace(go.Scatter3d(x=s.x, y=s.y, z=s.z, mode="markers",
                               marker=dict(size=1.5, color=color),
                               name=f"{label} ({len(s):,})"))

fig.update_layout(title="Left giant fiber: synapse locations")
fig.show()

### 1.3 Neurotransmitters of the inputs
Each synapse also comes with a predicted **neurotransmitter** (Eckstein et al., 2024): a probability for each of six transmitters, estimated from the image of the presynaptic site. Let's ask what kind of signals the GF receives:

In [ ]:
# pre=False, post=True → only synapses where the GF is postsynaptic (its inputs)
syn_nt = flywire.get_synapses(GF_LEFT, pre=False, post=True, transmitters=True)
syn_nt

In [ ]:
# add a column with the most likely transmitter per synapse
nts = ["acetylcholine", "gaba", "glutamate", "dopamine", "serotonin", "octopamine"]
# idxmax(axis=1): for each synapse (row), the name of the column with the highest probability
syn_nt["nt"] = syn_nt[nts].idxmax(axis=1)
syn_nt

In [ ]:
# Fraction of synapses per transmitter (0 for transmitters that don't occur), sorted for the plot
nt_frac = syn_nt.nt.value_counts(normalize=True).reindex(nts).fillna(0).sort_values()

fig, ax = plt.subplots(figsize=(5, 2.8))
ax.barh(nt_frac.index, nt_frac.values * 100, color="#2a78d6", height=0.7)
for y, v in enumerate(nt_frac.values * 100):
    ax.text(v + 1, y, f"{v:.0f}%", va="center", fontsize=9)
ax.set_xlabel("% of input synapses")
ax.set_title("Predicted transmitter of GF input synapses")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

In the fly, **acetylcholine** is the main *excitatory* transmitter, while **GABA** and (usually) **glutamate** are *inhibitory*. So the GF receives a mix of excitation and inhibition, dominated by excitation.

### 1.4 A common pitfall: mixing materializations
We can also get the synapses between particular neurons. However, root IDs can change across materializations, and IDs that never existed at the *same* time cannot be queried together. Below we combine the GF's root ID at v630 with its ID at v783, two IDs for the same neuron (one old, one current).

**This is expected to fail!** (We catch the error so the notebook keeps running.)

In [ ]:
# The giant fiber's root ID at materialization 630 (see Part 1)
root_630 = flywire.locs_to_segments([[110241, 59613, 3393]], timestamp="mat_630")[0]
print("GF root at v630:", root_630, "| at v783:", GF_LEFT)

# Querying IDs from two different materializations together fails: catch and print the error
try:
    syn = flywire.get_synapses([root_630, GF_LEFT])
except Exception as e:
    print("\nError:", e)

(If no error appeared, the GF was not edited between v630 and v783, so both IDs are the same. Lucky!)

The fix is to bring all IDs to the same point in time, e.g. with `flywire.update_ids()` (Part 1), or to query each ID at the materialization it belongs to, e.g. `flywire.get_synapses(root_630, materialization=630)`.

---
## 2. Connections: from synapses to a wiring diagram

### 2.1 Counting synapses
A **connection** (or *edge*) between two neurons is simply the number of synapses between them, also called the connection's **weight**. We can compute it ourselves from the synapse table:

In [ ]:
# Count the input synapses per presynaptic partner
inputs.groupby("pre").size().sort_values(ascending=False)

`get_connectivity` does exactly this (and more) for us. It returns a table of edges with `pre`, `post` and `weight`, covering both up- and downstream partners:

In [ ]:
# One row per connection: presynaptic neuron, postsynaptic neuron and number of synapses (weight)
cn = flywire.get_connectivity(GF_LEFT)
cn.head(10)

How many partners does the GF have? That depends on what you count as a "real" connection. Automatic synapse detection makes errors, so single synapses are often noise. Many studies only consider connections with **≥ 5 synapses**:

In [ ]:
up = cn[cn.post == GF_LEFT].copy()     # upstream partners (inputs)
down = cn[cn.pre == GF_LEFT].copy()    # downstream partners (outputs)

# (up.weight >= thr).sum() counts the partners connected by at least `thr` synapses
for thr in [1, 5, 10]:
    print(f"weight ≥ {thr:>2}: {(up.weight >= thr).sum():>4} upstream partners, "
          f"{(down.weight >= thr).sum():>3} downstream partners")

### 2.2 Who talks to the giant fiber?
A list of root IDs isn't very insightful. Let's add the **cell type** of each partner and sum the synapses per type:

In [ ]:
# adding presynaptic cell type
# map() looks up each presynaptic ID in type_of; neurons without a label become 'untyped'
up["type"] = up.pre.map(type_of).fillna("untyped")
up

In [ ]:
# Group the upstream connections by the type of the presynaptic neuron
by_type = up.groupby("type")
# Per type: number of different neurons and total number of synapses, largest first
by_type = (by_type.agg(n_neurons=("pre", "nunique"), synapses=("weight", "sum")).sort_values("synapses", ascending=False))
# Each type's share of all input synapses, in percent
by_type["% of input"] = (100 * by_type.synapses / by_type.synapses.sum()).round(1)
by_type.head(15)

In [ ]:
top = by_type.head(15).iloc[::-1]   # reverse so the biggest bar is on top
colors = [COLORS.get(t, GRAY) for t in top.index]

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top.index, top["% of input"], color=colors, height=0.7)
for y, (v, n) in enumerate(zip(top["% of input"], top.n_neurons)):
    ax.text(v + 0.3, y, f"{v:.1f}%  (n={n})", va="center", fontsize=8)
ax.set_xlabel("% of all input synapses to the left GF")
ax.set_title("Top 15 input types of the left giant fiber")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

The two looming-detector types we saw in Part 1, **LPLC2** and **LC4**, are among the biggest inputs. Each type contributes through many neurons, and each neuron views a different patch of the visual field.

> **Your turn (2 min):** Repeat the analysis for the **downstream** partners (`down`) in the brain. Which types does the GF talk to?

<details><summary>Show solution</summary>

```python
down["type"] = down.post.map(type_of).fillna("untyped")
down.groupby("type").weight.sum().sort_values(ascending=False).head(10)
```
You'll find comparatively few synapses, mostly onto other descending neurons and neurons near the GF axon. The real targets are in the nerve cord (see 1.1).
</details>

In [ ]:
# your code here

### 2.3 Neuron by neuron: the adjacency matrix
So far we pooled all neurons of a type. Is every single LC4 and LPLC2 neuron connected to the GF, or only a few of them? `get_adjacency` returns a matrix of synapse counts between a set of **source** neurons (rows) and **target** neurons (columns):

In [ ]:
# Rows: left LC4 and LPLC2 neurons; columns: the two giant fibers; values: number of synapses
adj = flywire.get_adjacency(sources=NC(type=["LC4", "LPLC2"], side="left"), targets=[GF_LEFT, GF_RIGHT], square=False)
adj

In [ ]:
# The cell type of each row (source neuron)
src_type = adj.index.map(type_of)

# .ge(5) marks neurons with ≥ 5 synapses; the mean of these True/False values per type is a fraction
summary = pd.DataFrame({
    "n_neurons": adj.groupby(src_type).size(),
    "→ left GF: % with ≥5 syn": adj[GF_LEFT].ge(5).groupby(src_type).mean().mul(100).round(0),
    "→ left GF: median syn": adj[GF_LEFT].groupby(src_type).median(),
    "→ right GF: % with ≥5 syn": adj[GF_RIGHT].ge(5).groupby(src_type).mean().mul(100).round(0),
})
summary

Let's look at the full distribution: one dot per neuron, showing how many synapses it makes onto the left GF.

In [ ]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(4.5, 4))
for i, t in enumerate(["LC4", "LPLC2"]):
    w = adj.loc[src_type == t, GF_LEFT]
    ax.scatter(i + rng.uniform(-0.15, 0.15, len(w)), w, s=12, color=COLORS[t], alpha=0.8)
    ax.hlines(w.median(), i - 0.25, i + 0.25, color="k", lw=2)
ax.set_xticks([0, 1], ["LC4", "LPLC2"])
ax.set_xlim(-0.6, 1.6)
ax.set_ylabel("synapses onto left GF (per neuron)")
ax.set_title("Each dot = one neuron (line = median)")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

### 2.4 Where do the inputs land?
Synapse positions let us ask questions that a simple wiring diagram can't answer, for example *where* on the GF each input type connects. Earlier work (Ache et al., 2019) reported that LC4 and LPLC2 contact **different parts** of the GF dendrite. Can we see this in FlyWire?

In [ ]:
cn_tab = sk.connectors.copy()
# The cell type of the partner neuron of each synapse
cn_tab["partner_type"] = cn_tab.partner_id.map(type_of)
# Input synapses (type 'post') coming from LC4 or LPLC2 neurons
vis_in = cn_tab[(cn_tab.type == "post") & cn_tab.partner_type.isin(["LC4", "LPLC2"])]

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, view, (c1, c2) in zip(axes, [("x", "-y"), ("x", "-z")], [("x", "y"), ("x", "z")]):
    navis.plot2d(sk, color=GRAY, lw=0.4, ax=ax, view=view)
    handles = []
    for t in ["LC4", "LPLC2"]:
        s = vis_in[vis_in.partner_type == t]
        h = ax.scatter(s[c1], s[c2], s=3, color=COLORS[t], label=f"{t} ({len(s):,} syn)", zorder=10)
        handles.append(h)
    # zoom in on the dendrite
    pad = 15_000  # nm
    ax.set_xlim(vis_in[c1].min() - pad, vis_in[c1].max() + pad)
    lo, hi = vis_in[c2].min() - pad, vis_in[c2].max() + pad
    ax.set_ylim(hi, lo)  # keep the inverted axis of the view
    ax.locator_params(nbins=4)
axes[0].set_title("frontal view")
axes[1].set_title("top view")
fig.legend(handles=handles, loc="upper center", ncol=2, markerscale=4, frameon=False)
plt.tight_layout(rect=(0, 0, 1, 0.93))

We can put a number on it: how far apart are the "centers of mass" of the two synapse clouds, compared with how spread out each cloud is?

In [ ]:
xyz = ["x", "y", "z"]
# Mean x/y/z position of each type's synapses (nm → µm)
centroids = vis_in.groupby("partner_type")[xyz].mean() / 1000   # µm
# Standard deviation along x, y and z, averaged: how spread out each type's synapses are
spread = vis_in.groupby("partner_type")[xyz].std().mean(axis=1) / 1000
print("Centroids (µm):\n", centroids.round(1), "\n")
# np.linalg.norm gives the straight-line distance between the two centroids
print("Distance between centroids:", round(float(np.linalg.norm(centroids.loc["LC4"] - centroids.loc["LPLC2"])), 1), "µm")
print("Average spread (SD) per type:\n", spread.round(1))

Why does synapse location matter?

---
## 3. Small exercises

### Exercise 1 **Left vs right.** 

Repeat Section 2.2 for `GF_RIGHT`. Are the top input types the same as for the left GF? Are the percentages similar? (This is a test of *stereotypy*: is the brain wired the same way on both sides?)

<details><summary>Show solution for 1</summary>

```python
cn_r = flywire.get_connectivity(GF_RIGHT, downstream=False)
up_r = cn_r[cn_r.post == GF_RIGHT].copy()
up_r["type"] = up_r.pre.map(type_of).fillna("untyped")
by_type_r = up_r.groupby("type").weight.sum().sort_values(ascending=False)
pd.concat({"left GF %": (100 * by_type.synapses / by_type.synapses.sum()).round(1),
           "right GF %": (100 * by_type_r / by_type_r.sum()).round(1)}, axis=1).head(15)
```
</details>

In [ ]:
# your code here

### Exercise 2 **Strongest per neuron.** 

In `by_type`, LPLC2 may give the most synapses in total, but it also has the most neurons. Which input type makes the most synapses **per neuron**?

<details><summary>Show solution for 2</summary>

```python
by_type["syn_per_neuron"] = by_type.synapses / by_type.n_neurons
by_type.sort_values("syn_per_neuron", ascending=False).head(10)
```
</details>

In [ ]:
# your code here

---
## Wrap-up

* A **synapse** table lists individual synapses: who sends (`pre`), who receives (`post`), where, and how confident the detection is.
* A **connection** is a count of synapses between two neurons (`get_connectivity`, `get_adjacency`). Thresholds matter!
* Mapping root IDs to **cell types** turns a list of numbers into biology: the giant fiber listens to looming-detecting visual neurons (LC4, LPLC2), and these inputs land on different parts of its dendrite.
* Every connectome has **blind spots**: the volume's boundaries (no nerve cord), electrical synapses (gap junctions), and prediction errors (neurotransmitters).

### Further resources
* [FlyWire Codex](https://codex.flywire.ai): browse neurons, cell types and connections in the browser
* [fafbseg tutorials](https://fafbseg-py.readthedocs.io/en/latest/source/tutorials/flywire_intro.html) and [navis tutorials](https://navis-org.github.io/navis/)
* Dorkenwald et al. (2024) *Neuronal wiring diagram of an adult brain*. Nature
* Schlegel et al. (2024) *Whole-brain annotation and multi-connectome cell typing of Drosophila*. Nature
* Ache et al. (2019) *Neural basis for looming size and velocity encoding in the Drosophila giant fiber escape pathway*. Current Biology